## Loading the ontology

In [56]:
from pathlib import Path

from rdflib import OWL, RDF, RDFS, Graph, Namespace, URIRef

BASE_DIR = Path.cwd().parent
ONTOLOGY_PATH = BASE_DIR / "data" / "wittgenstein.owl"

if not ONTOLOGY_PATH.exists():
    raise FileNotFoundError(f"Ontology file not found: {ONTOLOGY_PATH}")

graph = Graph()
graph.parse(str(ONTOLOGY_PATH))

WAB = Namespace("http://purl.org/wittgensteinsource/ont/")
print(f"Loaded triples: {len(graph):,}")


Loaded triples: 749,032


## Inspecting the relevant class

In [57]:
part_class = WAB.Part

print("Class:", part_class)
for predicate in (RDFS.label, RDFS.comment, RDFS.subClassOf):
    for value in graph.objects(part_class, predicate):
        print(f"{predicate.split('#')[-1].split('/')[-1]}: {value}")

part_instances = list(graph.subjects(RDF.type, part_class))
print(f"Number of Part instances: {len(part_instances):,}")

Class: http://purl.org/wittgensteinsource/ont/Part
label: Part
comment: Part
subClassOf: http://purl.org/wittgensteinsource/ont/WittgensteinSource
Number of Part instances: 7,881


## inspect a small sample of Part instances

In [58]:
label_by_instance = {
    subject: str(label)
    for subject, label in graph.subject_objects(RDFS.label)
}

relation_names = {
    WAB.isPartOf,
    WAB.isWorkPublishedFrom,
    WAB.isPublishedInWork,
    WAB.hasPart,
}

seen_work_labels = set()
shown = 0

for instance in part_instances:
    label = label_by_instance.get(instance, "")
    if not label:
        continue

    work_label = label.split(" (")[0]
    if work_label in seen_work_labels:
        continue

    seen_work_labels.add(work_label)
    print(f"\nLabel: {label}")
    print(f"URI: {instance}")

    for predicate in relation_names:
        for obj in graph.objects(instance, predicate):
            print(f"  {predicate.split('/')[-1]} -> {obj}")
        for subject in graph.subjects(predicate, instance):
            print(f"  incoming {predicate.split('/')[-1]} <- {subject}")

    shown += 1
    if shown >= 10:
        break

print(f"\nDistinct work labels sampled: {shown}")



Label: Prototractatus (PT, eds. 1971 McGuinness, Nyberg et von Wright) 0
URI: http://purl.org/wittgensteinsource/ont/instances/W-PT%3A%200
  isPartOf -> http://purl.org/wittgensteinsource/ont/instances/W-PT
  incoming isPublishedInWork <- http://wittgensteinsource.org/Ms-104%2CIII%5B1%5D_d
  incoming isPublishedInWork <- http://wittgensteinsource.org/Ms-104%2CV%5B1%5D_d
  incoming isPublishedInWork <- http://wittgensteinsource.org/Ms-104%2CVI%5B1%5D_d
  incoming isPublishedInWork <- http://wittgensteinsource.org/Ms-104%2C1%5B1%5D_d
  incoming isPublishedInWork <- http://wittgensteinsource.org/Ms-104%2C119%5B1%5Det120%5B1%5Det121%5B1%5D_d
  isWorkPublishedFrom -> http://wittgensteinsource.org/Ms-104%2CIII%5B1%5D_d
  isWorkPublishedFrom -> http://wittgensteinsource.org/Ms-104%2CV%5B1%5D_d
  isWorkPublishedFrom -> http://wittgensteinsource.org/Ms-104%2CVI%5B1%5D_d
  isWorkPublishedFrom -> http://wittgensteinsource.org/Ms-104%2C1%5B1%5D_d
  isWorkPublishedFrom -> http://wittgensteinsource

## Inspecting proposition 6.23

In [59]:
target_label = "Tractatus logico-philosophicus (TLP, 1922 et 1933) 6.23"
target_matches = [
    subject for subject, label in graph.subject_objects(RDFS.label)
    if str(label) == target_label
]

print("Matches:", len(target_matches))
for target in target_matches:
    print("WAB individual:", target)
    for predicate in (RDF.type, RDFS.label, WAB.isPartOf, WAB.isWorkPublishedFrom):
        for obj in graph.objects(target, predicate):
            print(f"  {predicate.split('/')[-1]} -> {obj}")
    for subject in graph.subjects(WAB.hasPart, target):
        print(f"  incoming hasPart <- {subject}")


Matches: 1
WAB individual: http://purl.org/wittgensteinsource/ont/instances/W-TLP%3A%206.23
  22-rdf-syntax-ns#type -> http://purl.org/wittgensteinsource/ont/Part
  rdf-schema#label -> Tractatus logico-philosophicus (TLP, 1922 et 1933) 6.23
  isPartOf -> http://purl.org/wittgensteinsource/ont/instances/W-TLP
  isWorkPublishedFrom -> http://wittgensteinsource.org/Ts-202%2C46r%5B8%5D_d
  isWorkPublishedFrom -> http://wittgensteinsource.org/Ts-203%2C50r%5B3%5D_d
  isWorkPublishedFrom -> http://wittgensteinsource.org/Ts-204%2C46r%5B8%5D_d
  incoming hasPart <- http://purl.org/wittgensteinsource/ont/instances/W-TLP


## Checking language and text sources

In [60]:
from collections import Counter

from rdflib import URIRef

language_property = URIRef(
    "http://purl.org/wittgensteinsource/ont/language"
)

language_counts = Counter(
    str(language)
    for language in graph.objects(None, language_property)
)

for language, count in language_counts.most_common():
    print(f"{count:>8} | {language}")

   52454 | http://purl.org/wittgensteinsource/ont/deu
    2172 | http://purl.org/wittgensteinsource/ont/eng


In [61]:
from rdflib import Literal

english_literals = []

for subject, predicate, obj in graph:
    if isinstance(obj, Literal) and obj.language:  # noqa: SIM102
        if obj.language.lower().startswith("en"):
            english_literals.append(
                (subject, predicate, obj)
            )

print("English-tagged literals:", len(english_literals))

for subject, predicate, obj in english_literals[:30]:
    print("\nSubject:", subject)
    print("Property:", predicate)
    print("Text:", obj)

English-tagged literals: 1

Subject: http://purl.org/wittgensteinsource/ont/hasVariant
Property: http://www.w3.org/2000/01/rdf-schema#comment
Text: A property for stating that a Bemerkung is or contains a variant of another 
Bemerkung.


In [62]:
import requests
from bs4 import BeautifulSoup
from rdflib import RDFS, URIRef

document_label = "Ts-202,46r[8]"

document_uri = next(
    graph.subjects(RDFS.label, None)
)

documents = [
    subject
    for subject in graph.subjects(RDFS.label, None)
    if str(graph.value(subject, RDFS.label)) == document_label
]

print("Number of documents found:", len(documents))

if documents:
    document_uri = documents[0]

    html_property = URIRef(
        "http://purl.org/wittgensteinsource/ont/hasHtmlView"
    )

    html_url = graph.value(document_uri, html_property)

    print("HTML URL:", html_url)

    if html_url:
        response = requests.get(str(html_url), timeout=30)
        print("Status code:", response.status_code)

        soup = BeautifulSoup(response.text, "html.parser")
        print(soup.get_text(" ", strip=True)[:3000])
    else:
        print("HTML URL not found.")
else:
    print("Document not found.")

Number of documents found: 1
HTML URL: https://wab.uib.no/nachlass/bemerkung/Ts-202%2C46r%5B8%5D_linear.html
Status code: 200
Wittgenstein Nachlass Ts-202: Logisch-Philosophische Abhandlung [so-called Engelmann TS of the Tractatus] (Bodl, MS. German d. 6)  - The Bergen Nachlass Edition in Interactive Dynamic Presentation (IDP). [http://wittgensteinonline.no/://wab.uib.no/transform/wab.php?modus=opsjoner]. Edited by the Wittgenstein Archives at the University of Bergen (WAB) under the direction of Alois Pichler. Bergen: Wittgenstein Archives at the University of Bergen 2016-. 6.23 Wenn zwei Ausdrücke durch das Gleichheitszeichen verbunden werden, so 
                           heißt das, sie sind durch einander ersetzbar. 
                           Ob dies aber der Fall  ist, muß sich an den beiden Ausdrücken selbst
                           zeigen. Es charakterisiert die logische Form zweier Ausdrücke, daß sie durch 
                           einander ersetzbar sind.


In [63]:
from rdflib import URIRef

proposition_uri = URIRef(
    "http://purl.org/wittgensteinsource/ont/instances/W-TLP%3A%206.23"
)

language_property = URIRef(
    "http://purl.org/wittgensteinsource/ont/language"
)

html_property = URIRef(
    "http://purl.org/wittgensteinsource/ont/hasHtmlView"
)

documents = list(
    graph.objects(proposition_uri, URIRef(
        "http://purl.org/wittgensteinsource/ont/isWorkPublishedFrom"
    ))
)

print("Number of documents:", len(documents))

for document in documents:
    language = graph.value(document, language_property)
    html_url = graph.value(document, html_property)

    print("\nDOCUMENT:", document)
    print("LANGUAGE:", language)
    print("HTML URL:", html_url)

Number of documents: 3

DOCUMENT: http://wittgensteinsource.org/Ts-202%2C46r%5B8%5D_d
LANGUAGE: http://purl.org/wittgensteinsource/ont/deu
HTML URL: https://wab.uib.no/nachlass/bemerkung/Ts-202%2C46r%5B8%5D_linear.html

DOCUMENT: http://wittgensteinsource.org/Ts-203%2C50r%5B3%5D_d
LANGUAGE: http://purl.org/wittgensteinsource/ont/deu
HTML URL: https://wab.uib.no/nachlass/bemerkung/Ts-203%2C50r%5B3%5D_linear.html

DOCUMENT: http://wittgensteinsource.org/Ts-204%2C46r%5B8%5D_d
LANGUAGE: http://purl.org/wittgensteinsource/ont/deu
HTML URL: https://wab.uib.no/nachlass/bemerkung/Ts-204%2C46r%5B8%5D_linear.html


In [64]:
import requests
from bs4 import BeautifulSoup
from rdflib import URIRef

html_property = URIRef(
    "http://purl.org/wittgensteinsource/ont/hasHtmlView"
)

for document in documents:
    html_url = graph.value(document, html_property)

    response = requests.get(str(html_url), timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(" ", strip=True)

    print("\nDOCUMENT:", document)
    print("HTML URL:", html_url)
    print("TEXT:", text[-1500:])


DOCUMENT: http://wittgensteinsource.org/Ts-202%2C46r%5B8%5D_d
HTML URL: https://wab.uib.no/nachlass/bemerkung/Ts-202%2C46r%5B8%5D_linear.html
TEXT: Wittgenstein Nachlass Ts-202: Logisch-Philosophische Abhandlung [so-called Engelmann TS of the Tractatus] (Bodl, MS. German d. 6)  - The Bergen Nachlass Edition in Interactive Dynamic Presentation (IDP). [http://wittgensteinonline.no/://wab.uib.no/transform/wab.php?modus=opsjoner]. Edited by the Wittgenstein Archives at the University of Bergen (WAB) under the direction of Alois Pichler. Bergen: Wittgenstein Archives at the University of Bergen 2016-. 6.23 Wenn zwei Ausdrücke durch das Gleichheitszeichen verbunden werden, so 
                           heißt das, sie sind durch einander ersetzbar. 
                           Ob dies aber der Fall  ist, muß sich an den beiden Ausdrücken selbst
                           zeigen. Es charakterisiert die logische Form zweier Ausdrücke, daß sie durch 
                           einander ersetzba

In [65]:
from rdflib import URIRef

target = URIRef(
    "http://purl.org/wittgensteinsource/ont/instances/W-TLP%3A%206.23"
)

print("Triples where the target is the subject:")
for triple in graph.triples((target, None, None)):
    print(triple)

print("Triples where the target is the object:")
for triple in graph.triples((None, None, target)):
    print(triple)

Triples where the target is the subject:
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/instances/W-TLP%3A%206.23'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/Part'))
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/instances/W-TLP%3A%206.23'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#label'), rdflib.term.Literal('Tractatus logico-philosophicus (TLP, 1922 et 1933) 6.23'))
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/instances/W-TLP%3A%206.23'), rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/isPartOf'), rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/instances/W-TLP'))
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/instances/W-TLP%3A%206.23'), rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/isWorkPublishedFrom'), rdflib.term.URIRef('http://wittgensteinsource.org/Ts-202%2C46r%5B8%5D_d'))
(rdflib.term.

In [66]:
from rdflib import OWL, RDF, RDFS, Namespace

WAB = Namespace("http://purl.org/wittgensteinsource/ont/")

part_class = WAB.Part

print("Statements about the class Part:")
for triple in graph.triples((part_class, None, None)):
    print(triple)

print("Statements referring to Part:")
for triple in graph.triples((None, None, part_class)):
    print(triple)

Statements about the class Part:
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/Part'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#Class'))
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/Part'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#comment'), rdflib.term.Literal('Part', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/Part'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/WittgensteinSource'))
(rdflib.term.URIRef('http://purl.org/wittgensteinsource/ont/Part'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#label'), rdflib.term.Literal('Part', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#string')))
Statements referring to Part:
(rdflib.term.BNode('Nddbb028f0bf4426aae6

exploring ontology classes

In [67]:
from collections import Counter

from rdflib import OWL, RDF, RDFS

class_counts = Counter()

for subject, class_uri in graph.subject_objects(RDF.type):
    if (
        (class_uri, RDF.type, OWL.Class) in graph
        or (class_uri, RDF.type, RDFS.Class) in graph
    ):
        class_counts[class_uri] += 1

print("Number of classes with instances:", len(class_counts))

for class_uri, count in class_counts.most_common(20):
    label = graph.value(class_uri, RDFS.label)
    print(f"{label or class_uri}: {count}")

Number of classes with instances: 10
Nachlass Bemerkung: 54626
MS: 32189
TS: 22600
Part: 7881
Date: 2206
WittgensteinExternalSource: 428
Person: 343
Work: 71
Text subgenre: 13
Language: 4


In [68]:
from rdflib import RDFS

for class_uri, count in class_counts.most_common():
    label = graph.value(class_uri, RDFS.label)
    comment = graph.value(class_uri, RDFS.comment)

    print("\nCLASS:", label or class_uri)
    print("URI:", class_uri)
    print("COUNT:", count)
    print("COMMENT:", comment)


CLASS: Nachlass Bemerkung
URI: http://purl.org/wittgensteinsource/ont/Bemerkung
COUNT: 54626
COMMENT: Bemerkung, subclass of Wittgenstein Primary source: Corresponding roughly to a
            single remark in a Wittgenstein primary source, identified by the
            Wittgenstein Archives through a “siglum” which is composed of item identifier, page
            identifier(s) and position identifier(s). Example of instance: Ts-310,21[2]et22[1]
        

CLASS: MS
URI: http://purl.org/wittgensteinsource/ont/MS
COUNT: 32189
COMMENT: MS, subclass of Wittgenstein Primary Source: A Wittgenstein primary source, handwritten. Example of instance: Ms-115; Ms-139a 

CLASS: TS
URI: http://purl.org/wittgensteinsource/ont/TS
COUNT: 22600
COMMENT: TS, subclass of Wittgenstein Primary source: A Wittgenstein primary source, typed. Example of instance: Ts-213; Ts-310 

CLASS: Part
URI: http://purl.org/wittgensteinsource/ont/Part
COUNT: 7881
COMMENT: Part

CLASS: Date
URI: http://purl.org/wittgenstei